In [1]:
# ============================================================
# LongPort Bull Put Spread — Test Order
# Fill in YOUR credentials before running
# ============================================================

APP_KEY = "6e73506603d52be23b36cc884ca6fd8b"
APP_SECRET = "66785f824db9072cf288d52ea0a1ca6649f3fddb61d4f0a01534277ce2500c58"  
ACCESS_TOKEN = "m_eyJhbGciOiJSUzI1NiIsImtpZCI6ImQ5YWRiMGIxYTdlNzYxNzEiLCJ0eXAiOiJKV1QifQ.eyJpc3MiOiJsb25nYnJpZGdlIiwic3ViIjoiYWNjZXNzX3Rva2VuIiwiZXhwIjoxNzg0Njc5MzE3LCJpYXQiOjE3NzY5MDMzMTYsImFrIjoiNmU3MzUwNjYwM2Q1MmJlMjNiMzZjYzg4NGNhNmZkOGIiLCJhYWlkIjoyMTMxNDk3NSwiYWMiOiJsYl9zZyIsIm1pZCI6MjY4NjE0MDcsInNpZCI6IjMwK0xWMlpFSHhaY01qeHl3NEpIeFE9PSIsImJsIjozLCJ1bCI6MCwiaWsiOiJsYl9zZ18yMTMxNDk3NSJ9.5e4x5MHtSwXQ7Uw4e_kUUDjA-SPrbsP8YWzmtspIWRtpn7KaF00Z-fPOUTeLITfoWLl1i1JnUdix8Ka2IPbaC8t6Ns6VPvwKLpFs0Opv86Bx-RdVZiGwZ92prs0GNh47LVUTpOpoWvPYa0RvNTuyQNTkrx2j-fMXG1jd037dlUqgRp-mxjA5ePqTtC-VUhmkOWg3mGCfrGmRJR45ZVo-Q7mZuDCVm_9eGej5ZdwobOL49-rgStFXs3FKaYNY67Rtu-ZKL2Segi9-P3DMac80OrwEEue6Nf_2_Bg0i4HCuEQBG_ynMIzqMydcO5FLLcv-9TlG6-Hapfdwnd49aPGhkL2opDNlZyRKemXcKtyfCk1hf_0VL5BM1T4fAvNVlosDYQEp2Uk2UShuMiUWueKkNy49gjbbClWTx5qalMGJXZ37U6aJgMc75kBH9K2PHH1DWyEKb4VNv_hYwpx_RgmABytNGHEGDRH5Slx7eW6I1o6yQcjq0yl_uw9oMptFWDt5wYVdU2mKH4tdm-q0OYCO1OiSI9ji-vkXSBGy3PEavvsqq9u3yfl87IfcESmcrp5V7Rfco2CXmwHoyZziI1ad_oRC3wygU4Gx8utBgmbWhEOWkW6Po9j6Z5GEPqWHqpxwp6S69Xxl4Z1I_en0GuWffVIQojPFTnH9SbHEKqY3eio"

# Order config — SPY bull put spread
SYMBOL_SHORT = "SPY240119P00460000"  # SELL this put (higher strike)
SYMBOL_LONG  = "SPY240119P00455000"  # BUY this put  (lower strike)

# Use paper trading? Set True for testing
PAPER_TRADE = True  # Change to False for live

QTY = 1  # number of contracts

print("Config loaded ✅")
print(f"Short Put: {SYMBOL_SHORT}")
print(f"Long Put:  {SYMBOL_LONG}")
print(f"Paper:     {PAPER_TRADE}")

Config loaded ✅
Short Put: SPY240119P00460000
Long Put:  SPY240119P00455000
Paper:     True


In [2]:
from longport.openapi import Config, TradeContext, QuoteContext
from longport.openapi import OrderType, OrderSide, TimeInForceType
import decimal

# Connect
cfg = Config(
    app_key=APP_KEY,
    app_secret=APP_SECRET,
    access_token=ACCESS_TOKEN,
)

# Test quote context first
print("Connecting to LongPort...")
try:
    quote_ctx = QuoteContext(cfg)
    print("✅ QuoteContext connected")
except Exception as e:
    print(f"❌ QuoteContext failed: {e}")

# Test trade context
try:
    trade_ctx = TradeContext(cfg)
    print("✅ TradeContext connected")
except Exception as e:
    print(f"❌ TradeContext failed: {e}")
    raise

# Check account balance
try:
    balances = trade_ctx.account_balance()
    for b in balances:
        print(f"\n💰 Account: {b.currency}")
        print(f"   Cash:     {b.cash_infos}")
        print(f"   Net assets: {b.net_assets}")
except Exception as e:
    print(f"⚠️  Could not fetch balance: {e}")

Connecting to LongPort...
✅ QuoteContext connected
✅ TradeContext connected

💰 Account: SGD
   Cash:     [CashInfo { withdraw_cash: 2317.61, available_cash: 2317.61, frozen_cash: 0.00, settling_cash: 0.00, currency: "USD" }]
   Net assets: 2956.48


In [3]:
# Get current bid/ask for both option legs
print("Fetching option quotes...")

try:
    quotes = quote_ctx.quote([SYMBOL_SHORT, SYMBOL_LONG])
    for q in quotes:
        print(f"\n{q.symbol}")
        print(f"  Bid: {q.bid}  Ask: {q.ask}")
        print(f"  Last: {q.last_done}")
        print(f"  Volume: {q.volume}")
except Exception as e:
    print(f"❌ Quote fetch failed: {e}")
    print("Note: Option symbols must be in OCC format")
    print("Format: SPY YYMMDD [C/P] STRIKE*1000")
    print("Example: SPY240119P00460000 = SPY Jan 19 2024 Put 460")

Fetching option quotes...


In [4]:
# ============================================================
# STEP 1: Sell the higher-strike put (collect premium)
# This is the short leg of the bull put spread
# ============================================================

print(f"Placing SELL order: {SYMBOL_SHORT} x{QTY}")
print("=" * 50)

try:
    # For options we use LimitOrder with mid-price
    # First get the quote to calculate limit price
    quotes = quote_ctx.quote([SYMBOL_SHORT])
    q = quotes[0]
    bid = float(str(q.bid))
    ask = float(str(q.ask))
    mid = round((bid + ask) / 2, 2)
    
    print(f"Bid: ${bid:.2f}  Ask: ${ask:.2f}  Mid: ${mid:.2f}")
    
    # Place limit order at mid-price
    sell_resp = trade_ctx.submit_order(
        symbol=SYMBOL_SHORT,
        order_type=OrderType.LO,           # Limit Order
        side=OrderSide.Sell,               # SELL to open
        submitted_quantity=decimal.Decimal(QTY),
        submitted_price=decimal.Decimal(str(mid)),
        time_in_force=TimeInForceType.Day,
        remark="QuantX Bull Put Spread - Short Leg"
    )
    
    print(f"\n✅ SELL order placed!")
    print(f"   Order ID:  {sell_resp.order_id}")
    print(f"   Symbol:    {SYMBOL_SHORT}")
    print(f"   Side:      SELL")
    print(f"   Qty:       {QTY}")
    print(f"   Price:     ${mid:.2f}")
    
    SELL_ORDER_ID = str(sell_resp.order_id)

except Exception as e:
    print(f"❌ SELL order failed: {e}")
    SELL_ORDER_ID = None

Placing SELL order: SPY240119P00460000 x1
❌ SELL order failed: list index out of range


In [5]:
# Run this to get valid symbols for TODAY
from longport.openapi import Config, QuoteContext
import decimal
from datetime import date, timedelta

cfg = Config(
    app_key=APP_KEY,
    app_secret=APP_SECRET,
    access_token=ACCESS_TOKEN,
)
quote_ctx = QuoteContext(cfg)

# Get SPY current price
spy_quote = quote_ctx.quote(["SPY.US"])
spy_price = float(str(spy_quote[0].last_done))
print(f"SPY current price: ${spy_price:.2f}")

# Calculate strikes
short_strike = round(spy_price * 0.97 / 5) * 5
long_strike = short_strike - 5

print(f"Short Put strike: ${short_strike}")
print(f"Long Put strike:  ${long_strike}")

# Next Friday expiry
today = date.today()
days_to_friday = (4 - today.weekday()) % 7
if days_to_friday == 0:
    days_to_friday = 7
expiry = today + timedelta(days=days_to_friday)
exp_str = expiry.strftime("%y%m%d")
print(f"Expiry: {expiry} → {exp_str}")

# Build symbols
short_sym = f"SPY{exp_str}P{int(short_strike*1000):08d}"
long_sym  = f"SPY{exp_str}P{int(long_strike*1000):08d}"

print(f"\nUse these symbols:")
print(f'SYMBOL_SHORT = "{short_sym}"')
print(f'SYMBOL_LONG  = "{long_sym}"')

# Verify they exist
print("\nVerifying quotes...")
try:
    quotes = quote_ctx.quote([short_sym, long_sym])
    for q in quotes:
        print(f"  {q.symbol}: bid={q.bid} ask={q.ask}")
except Exception as e:
    print(f"Quote failed: {e}")
    print("Try next week's expiry or check symbol format")

SPY current price: $711.21
Short Put strike: $690
Long Put strike:  $685
Expiry: 2026-04-24 → 260424

Use these symbols:
SYMBOL_SHORT = "SPY260424P00690000"
SYMBOL_LONG  = "SPY260424P00685000"

Verifying quotes...


In [6]:
# Diagnose why quote is hanging/empty
import time

print("Testing quote for SPY option...")
print(f"Symbol: SPY260424P00690000")

try:
    # Try with .US suffix — LongPort may require market suffix
    symbols_to_try = [
        "SPY260424P00690000",
        "SPY260424P00690000.US", 
        "SPY 260424P00690000",  # with space
    ]
    
    for sym in symbols_to_try:
        print(f"\nTrying: {sym}")
        try:
            t0 = time.time()
            q = quote_ctx.quote([sym])
            elapsed = time.time() - t0
            print(f"  Result: {q}")
            print(f"  Time: {elapsed:.1f}s")
            if q:
                print(f"  Bid: {q[0].bid}  Ask: {q[0].ask}")
                print(f"  ✅ This format works!")
                break
        except Exception as e:
            print(f"  ❌ Error: {e}")

except Exception as e:
    print(f"Outer error: {e}")

# Also check what market data permissions we have
print("\n\nChecking account permissions...")
try:
    # Try a simple stock quote to confirm connection works
    stock = quote_ctx.quote(["SPY.US"])
    print(f"SPY.US quote works: {stock[0].last_done}")
    print("✅ Quote connection is fine — issue is options symbol format")
except Exception as e:
    print(f"❌ Even stock quote failed: {e}")

Testing quote for SPY option...
Symbol: SPY260424P00690000

Trying: SPY260424P00690000
  Result: []
  Time: 0.1s

Trying: SPY260424P00690000.US
  Result: []
  Time: 0.1s

Trying: SPY 260424P00690000
  Result: []
  Time: 0.1s


Checking account permissions...
SPY.US quote works: 711.210
✅ Quote connection is fine — issue is options symbol format


In [7]:
# ============================================================
# CELL 2 (FIXED) — Get option chain then quote correctly
# ============================================================
from longport.openapi import Config, QuoteContext, TradeContext
from longport.openapi import OrderType, OrderSide, TimeInForceType
import decimal
from datetime import date, timedelta

cfg = Config(
    app_key=APP_KEY,
    app_secret=APP_SECRET,
    access_token=ACCESS_TOKEN,
)
quote_ctx = QuoteContext(cfg)
trade_ctx = TradeContext(cfg)

# Step 1: Get SPY price
spy = quote_ctx.quote(["SPY.US"])
spy_price = float(str(spy[0].last_done))
print(f"SPY price: ${spy_price:.2f}")

# Step 2: Get available option expiry dates
print("\nFetching SPY option expiry dates...")
try:
    expiries = quote_ctx.option_chain_expiry_date_list("SPY.US")
    print(f"Available expiries: {[str(e) for e in expiries[:8]]}")
    
    # Pick the nearest expiry (tomorrow = 0DTE, or next Friday)
    next_expiry = expiries[0]  # nearest expiry
    print(f"\nUsing expiry: {next_expiry}")
    
except Exception as e:
    print(f"Error getting expiry dates: {e}")

# Step 3: Get the actual option chain for that expiry
print(f"\nFetching option chain for {next_expiry}...")
try:
    chain = quote_ctx.option_chain_info_by_date("SPY.US", next_expiry)
    
    # Find puts near current price
    puts = [o for o in chain if 'P' in str(o.symbol)]
    puts_sorted = sorted(puts, 
        key=lambda x: abs(float(str(x.strike_price)) - spy_price))
    
    print(f"\nPuts near ATM (${spy_price:.0f}):")
    for p in puts_sorted[:6]:
        print(f"  {p.symbol}  strike=${p.strike_price}")
    
    # Pick short put (~3% OTM) and long put ($5 below)
    short_put = None
    long_put = None
    target_short = spy_price * 0.97
    
    for p in sorted(puts, key=lambda x: float(str(x.strike_price)), reverse=True):
        strike = float(str(p.strike_price))
        if strike <= target_short and short_put is None:
            short_put = p
        if short_put and strike <= float(str(short_put.strike_price)) - 4:
            long_put = p
            break
    
    if short_put and long_put:
        print(f"\n✅ Selected legs:")
        print(f"  SELL: {short_put.symbol}  (strike ${short_put.strike_price})")
        print(f"  BUY:  {long_put.symbol}   (strike ${long_put.strike_price})")
        
        SYMBOL_SHORT = short_put.symbol
        SYMBOL_LONG = long_put.symbol
    else:
        print("❌ Could not find suitable strikes")

except Exception as e:
    print(f"Error getting option chain: {e}")

SPY price: $711.21

Fetching SPY option expiry dates...
Available expiries: ['2026-04-17', '2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23', '2026-04-24', '2026-04-27', '2026-04-28']

Using expiry: 2026-04-17

Fetching option chain for 2026-04-17...
Error getting option chain: 'builtins.StrikePriceInfo' object has no attribute 'symbol'


In [8]:
# ============================================================
# CELL 3 — Get real quotes using option_quote()
# ============================================================

print("Getting option quotes...")
try:
    quotes = quote_ctx.option_quote([SYMBOL_SHORT, SYMBOL_LONG])
    
    for q in quotes:
        bid = float(str(q.bid))
        ask = float(str(q.ask))
        mid = round((bid + ask) / 2, 2)
        iv = q.option_extend.implied_volatility if q.option_extend else "?"
        delta = "?"
        print(f"\n{q.symbol}")
        print(f"  Bid: ${bid:.2f}  Ask: ${ask:.2f}  Mid: ${mid:.2f}")
        print(f"  Last: {q.last_done}  IV: {iv}")
        print(f"  Open Interest: {q.option_extend.open_interest if q.option_extend else '?'}")

except Exception as e:
    print(f"❌ option_quote failed: {e}")
    print("Possible issues:")
    print("  1. No options quote permission in your LongPort account")
    print("  2. Market is closed (need 9:30am-4pm ET)")
    print("  3. Symbol not found")

Getting option quotes...
❌ option_quote failed: OpenApiException: (kind=ErrorKind.OpenApi, code=301604, trace_id=) no quote access
Possible issues:
  1. No options quote permission in your LongPort account
  2. Market is closed (need 9:30am-4pm ET)
  3. Symbol not found


In [9]:
# ============================================================
# CELL 4 — Place both legs
# ============================================================

def place_leg(ctx, quote_ctx, symbol, side, qty):
    """Place one leg of the spread at mid-price."""
    try:
        quotes = quote_ctx.option_quote([symbol])
        if not quotes:
            print(f"❌ No quote for {symbol}")
            return None
        
        q = quotes[0]
        bid = float(str(q.bid))
        ask = float(str(q.ask))
        
        if bid == 0 and ask == 0:
            print(f"❌ Zero bid/ask for {symbol} — market may be closed")
            return None
        
        # Use mid price, minimum $0.01
        mid = max(round((bid + ask) / 2, 2), 0.01)
        
        print(f"\n{'SELL' if side == OrderSide.Sell else 'BUY'} {symbol}")
        print(f"  Bid=${bid:.2f}  Ask=${ask:.2f}  → Limit @ ${mid:.2f}")
        
        resp = ctx.submit_order(
            symbol=symbol,
            order_type=OrderType.LO,
            side=side,
            submitted_quantity=decimal.Decimal(qty),
            submitted_price=decimal.Decimal(str(mid)),
            time_in_force=TimeInForceType.Day,
            remark="QuantX Bull Put Spread"
        )
        
        print(f"  ✅ Order ID: {resp.order_id}")
        return str(resp.order_id)
        
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        return None

# Place the spread
print("=" * 50)
print("PLACING BULL PUT SPREAD")
print("=" * 50)

QTY = 1

# Leg 1: Sell the short put (collect premium)
sell_id = place_leg(trade_ctx, quote_ctx, SYMBOL_SHORT, OrderSide.Sell, QTY)

# Small delay between legs
import time
time.sleep(1)

# Leg 2: Buy the long put (define max loss)  
buy_id = place_leg(trade_ctx, quote_ctx, SYMBOL_LONG, OrderSide.Buy, QTY)

print(f"\n{'='*50}")
print(f"SPREAD PLACED:")
print(f"  SELL {SYMBOL_SHORT} → Order: {sell_id}")
print(f"  BUY  {SYMBOL_LONG}  → Order: {buy_id}")

# Check net premium
if sell_id and buy_id:
    print(f"\n✅ Both legs placed successfully!")
    print(f"   Monitor in LongPort app or run Cell 5 to check status")
else:
    print(f"\n⚠️  One or both legs failed — check above errors")

PLACING BULL PUT SPREAD
  ❌ Failed: OpenApiException: (kind=ErrorKind.OpenApi, code=301604, trace_id=) no quote access
  ❌ Failed: OpenApiException: (kind=ErrorKind.OpenApi, code=301604, trace_id=) no quote access

SPREAD PLACED:
  SELL SPY240119P00460000 → Order: None
  BUY  SPY240119P00455000  → Order: None

⚠️  One or both legs failed — check above errors


In [10]:
# Check what attributes StrikePriceInfo actually has
chain = quote_ctx.option_chain_info_by_date("SPY.US", expiries[1])  # try April 24
print(f"Chain type: {type(chain)}")
print(f"Chain length: {len(chain)}")

if chain:
    first = chain[0]
    print(f"\nFirst item type: {type(first)}")
    print(f"First item dir(): {[a for a in dir(first) if not a.startswith('_')]}")
    print(f"First item repr: {first}")

Chain type: <class 'list'>
Chain length: 203

First item type: <class 'builtins.StrikePriceInfo'>
First item dir(): ['call_symbol', 'price', 'put_symbol', 'standard']
First item repr: StrikePriceInfo { price: 500, call_symbol: "SPY260420C500000.US", put_symbol: "SPY260420P500000.US", standard: true }


In [11]:
# Get valid symbols from chain (this works without quote permission)
chain = quote_ctx.option_chain_info_by_date("SPY.US", expiries[1])  # Apr 24

# Find puts near 3% OTM
target_short = spy_price * 0.97
puts = [(item.price, item.put_symbol) for item in chain 
        if item.put_symbol and float(str(item.price)) <= target_short]

# Sort to find closest OTM put
puts_sorted = sorted(puts, key=lambda x: abs(float(str(x[0])) - target_short))

print("Puts near 3% OTM:")
for price, sym in puts_sorted[:5]:
    print(f"  Strike ${price}  →  {sym}")

# Pick short and long legs
short_strike, SYMBOL_SHORT = puts_sorted[0]
long_candidates = [(p, s) for p, s in puts 
                   if float(str(p)) <= float(str(short_strike)) - 4]
long_strike, SYMBOL_LONG = sorted(long_candidates, 
    key=lambda x: float(str(x[0])), reverse=True)[0]

print(f"\nSelected:")
print(f"  SELL: {SYMBOL_SHORT}  (${short_strike})")
print(f"  BUY:  {SYMBOL_LONG}   (${long_strike})")
print(f"  Width: ${float(str(short_strike)) - float(str(long_strike)):.0f}")

Puts near 3% OTM:
  Strike $689  →  SPY260420P689000.US
  Strike $688  →  SPY260420P688000.US
  Strike $687  →  SPY260420P687000.US
  Strike $686  →  SPY260420P686000.US
  Strike $685  →  SPY260420P685000.US

Selected:
  SELL: SPY260420P689000.US  ($689)
  BUY:  SPY260420P685000.US   ($685)
  Width: $4


In [12]:
import decimal
import time

# ============================================================
# Place Bull Put Spread — fixed limit price (no quote needed)
# ============================================================

# Estimated prices (adjust based on market conditions)
# SPY at $711, selling $689 put ~4% OTM, 3 DTE
# Conservative estimates — will likely need price improvement
SELL_LIMIT_PRICE = "1.50"   # premium we collect for short put
BUY_LIMIT_PRICE  = "0.80"   # premium we pay for long put
# Net credit = $1.50 - $0.80 = $0.70 per share = $70 per contract

QTY = 1

print("=" * 55)
print("BULL PUT SPREAD — SPY Apr 20 2026")
print("=" * 55)
print(f"SELL {SYMBOL_SHORT}  @ ${SELL_LIMIT_PRICE}")
print(f"BUY  {SYMBOL_LONG}   @ ${BUY_LIMIT_PRICE}")
print(f"Net credit: ~${float(SELL_LIMIT_PRICE)-float(BUY_LIMIT_PRICE):.2f}")
print(f"Max profit: ~${(float(SELL_LIMIT_PRICE)-float(BUY_LIMIT_PRICE))*100:.0f}")
print(f"Max loss:   ~${(4 - float(SELL_LIMIT_PRICE)+float(BUY_LIMIT_PRICE))*100:.0f}")
print("=" * 55)

# Leg 1: SELL the short put
print("\nPlacing SELL leg...")
try:
    sell_resp = trade_ctx.submit_order(
        symbol=SYMBOL_SHORT,
        order_type=OrderType.LO,
        side=OrderSide.Sell,
        submitted_quantity=decimal.Decimal(QTY),
        submitted_price=decimal.Decimal(SELL_LIMIT_PRICE),
        time_in_force=TimeInForceType.Day,
        remark="QuantX BullPutSpread ShortLeg"
    )
    SELL_ORDER_ID = str(sell_resp.order_id)
    print(f"✅ SELL order placed: {SELL_ORDER_ID}")
except Exception as e:
    print(f"❌ SELL failed: {e}")
    SELL_ORDER_ID = None

time.sleep(1)

# Leg 2: BUY the long put
print("\nPlacing BUY leg...")
try:
    buy_resp = trade_ctx.submit_order(
        symbol=SYMBOL_LONG,
        order_type=OrderType.LO,
        side=OrderSide.Buy,
        submitted_quantity=decimal.Decimal(QTY),
        submitted_price=decimal.Decimal(BUY_LIMIT_PRICE),
        time_in_force=TimeInForceType.Day,
        remark="QuantX BullPutSpread LongLeg"
    )
    BUY_ORDER_ID = str(buy_resp.order_id)
    print(f"✅ BUY order placed: {BUY_ORDER_ID}")
except Exception as e:
    print(f"❌ BUY failed: {e}")
    BUY_ORDER_ID = None

# Summary
print("\n" + "=" * 55)
print("RESULT:")
print(f"  SELL {SYMBOL_SHORT}: {SELL_ORDER_ID or 'FAILED'}")
print(f"  BUY  {SYMBOL_LONG}:  {BUY_ORDER_ID or 'FAILED'}")

BULL PUT SPREAD — SPY Apr 20 2026
SELL SPY260420P689000.US  @ $1.50
BUY  SPY260420P685000.US   @ $0.80
Net credit: ~$0.70
Max profit: ~$70
Max loss:   ~$330

Placing SELL leg...
❌ SELL failed: OpenApiException: (kind=ErrorKind.OpenApi, code=602022, trace_id=7bd23fee6882b319755d41442f0702f3) Not allowed to trade expired options

Placing BUY leg...
❌ BUY failed: OpenApiException: (kind=ErrorKind.OpenApi, code=602022, trace_id=4663aa1d9fd838510d07ec68c82dd6a8) Not allowed to trade expired options

RESULT:
  SELL SPY260420P689000.US: FAILED
  BUY  SPY260420P685000.US:  FAILED


In [13]:
# Use NEXT available expiry (Apr 24 or later)
print("Available expiries:")
for i, e in enumerate(expiries[:8]):
    print(f"  [{i}] {e}")

# Find first expiry AFTER today
from datetime import date
today = date.today()
print(f"\nToday: {today}")

future_expiries = [e for e in expiries if str(e) > str(today)]
print(f"Future expiries: {future_expiries[:5]}")

next_exp = future_expiries[0]
print(f"\nUsing: {next_exp}")

# Get chain for this expiry
chain = quote_ctx.option_chain_info_by_date("SPY.US", next_exp)
target_short = spy_price * 0.97

puts = [(item.price, item.put_symbol) for item in chain 
        if item.put_symbol and float(str(item.price)) <= target_short]
puts_sorted = sorted(puts, key=lambda x: abs(float(str(x[0])) - target_short))

print(f"\nPuts near 3% OTM for {next_exp}:")
for price, sym in puts_sorted[:5]:
    print(f"  ${price}  →  {sym}")

short_strike, SYMBOL_SHORT = puts_sorted[0]
long_candidates = [(p,s) for p,s in puts 
                   if float(str(p)) <= float(str(short_strike)) - 4]
long_strike, SYMBOL_LONG = sorted(long_candidates,
    key=lambda x: float(str(x[0])), reverse=True)[0]

print(f"\nFinal selection:")
print(f"  SELL: {SYMBOL_SHORT}  (${short_strike})")
print(f"  BUY:  {SYMBOL_LONG}   (${long_strike})")

Available expiries:
  [0] 2026-04-17
  [1] 2026-04-20
  [2] 2026-04-21
  [3] 2026-04-22
  [4] 2026-04-23
  [5] 2026-04-24
  [6] 2026-04-27
  [7] 2026-04-28

Today: 2026-04-23
Future expiries: [datetime.date(2026, 4, 24), datetime.date(2026, 4, 27), datetime.date(2026, 4, 28), datetime.date(2026, 4, 29), datetime.date(2026, 4, 30)]

Using: 2026-04-24

Puts near 3% OTM for 2026-04-24:
  $689  →  SPY260424P689000.US
  $688  →  SPY260424P688000.US
  $687.5  →  SPY260424P687500.US
  $687  →  SPY260424P687000.US
  $686  →  SPY260424P686000.US

Final selection:
  SELL: SPY260424P689000.US  ($689)
  BUY:  SPY260424P685000.US   ($685)


In [14]:
import decimal, time

SELL_LIMIT_PRICE = "1.50"
BUY_LIMIT_PRICE  = "0.80"
QTY = 1

print("=" * 55)
print("BULL PUT SPREAD — SPY Apr 24 2026 (1DTE)")
print("=" * 55)
print(f"SELL {SYMBOL_SHORT} @ ${SELL_LIMIT_PRICE}")
print(f"BUY  {SYMBOL_LONG}  @ ${BUY_LIMIT_PRICE}")
print(f"Net credit: ~${float(SELL_LIMIT_PRICE)-float(BUY_LIMIT_PRICE):.2f}")
print(f"Max profit: ~${(float(SELL_LIMIT_PRICE)-float(BUY_LIMIT_PRICE))*100:.0f}")
print(f"Max loss:   ~${(4-(float(SELL_LIMIT_PRICE)-float(BUY_LIMIT_PRICE)))*100:.0f}")
print("=" * 55)

print("\nPlacing SELL leg...")
try:
    sell_resp = trade_ctx.submit_order(
        symbol=SYMBOL_SHORT,
        order_type=OrderType.LO,
        side=OrderSide.Sell,
        submitted_quantity=decimal.Decimal(QTY),
        submitted_price=decimal.Decimal(SELL_LIMIT_PRICE),
        time_in_force=TimeInForceType.Day,
        remark="QuantX BullPutSpread ShortLeg"
    )
    SELL_ORDER_ID = str(sell_resp.order_id)
    print(f"✅ SELL order: {SELL_ORDER_ID}")
except Exception as e:
    print(f"❌ SELL failed: {e}")
    SELL_ORDER_ID = None

time.sleep(1)

print("\nPlacing BUY leg...")
try:
    buy_resp = trade_ctx.submit_order(
        symbol=SYMBOL_LONG,
        order_type=OrderType.LO,
        side=OrderSide.Buy,
        submitted_quantity=decimal.Decimal(QTY),
        submitted_price=decimal.Decimal(BUY_LIMIT_PRICE),
        time_in_force=TimeInForceType.Day,
        remark="QuantX BullPutSpread LongLeg"
    )
    BUY_ORDER_ID = str(buy_resp.order_id)
    print(f"✅ BUY order: {BUY_ORDER_ID}")
except Exception as e:
    print(f"❌ BUY failed: {e}")
    BUY_ORDER_ID = None

print("\n" + "=" * 55)
print(f"SELL {SYMBOL_SHORT}: {SELL_ORDER_ID or 'FAILED'}")
print(f"BUY  {SYMBOL_LONG}:  {BUY_ORDER_ID or 'FAILED'}")

BULL PUT SPREAD — SPY Apr 24 2026 (1DTE)
SELL SPY260424P689000.US @ $1.50
BUY  SPY260424P685000.US  @ $0.80
Net credit: ~$0.70
Max profit: ~$70
Max loss:   ~$330

Placing SELL leg...
✅ SELL order: 1231867412714889216

Placing BUY leg...
✅ BUY order: 1231867417949376512

SELL SPY260424P689000.US: 1231867412714889216
BUY  SPY260424P685000.US:  1231867417949376512


In [15]:
# Check available order types for options
from longport.openapi import OrderType
print("Available OrderTypes:")
for ot in dir(OrderType):
    if not ot.startswith('_'):
        print(f"  {ot}")

Available OrderTypes:
  ALO
  AO
  ELO
  LIT
  LO
  MIT
  MO
  ODD
  SLO
  TSLPAMT
  TSLPPCT
  TSMAMT
  TSMPCT
  Unknown
